# 🚀 Thử nghiệm ứng viên 1: SeedVR2 (ByteDance Seed — ICLR 2026)

- **Mô hình:** SeedVR2-3B (Diffusion Transformer One-Step Video Restoration)
- **Giấy phép:** Apache 2.0 (Thương mại & Nguồn mở tự do)
- **Đặc điểm:** Khử nhiễu và làm nét video trong đúng 1 bước forward pass DiT, khắc phục hoàn toàn hiện tượng nhấp nháy (flicker) và giữ chi tiết khuôn mặt chân thực.
- **Yêu cầu phần cứng:** GPU Tesla T4 16GB (Google Colab) hoặc RTX 3090/4090.

Notebook này nạp 3 video test chuẩn (`clip_A`, `clip_B`, `clip_C`), thực hiện upscale 2x (720p → 1440p), đo đạc VRAM/thời gian và kết xuất video so sánh trực quan.

In [ ]:
# [1] Kiểm tra thông tin GPU
!nvidia-smi

In [ ]:
# [2] Thiết lập thư mục và tải dự án Eco Video Upscaler
%cd /content
!rm -rf /content/upscale_video
!git clone https://github.com/mloc07/upscale_video.git
%cd /content/upscale_video

# Cài đặt các thư viện lõi, webui và media cần thiết
!pip install -q gradio onnxruntime-gpu diffusers accelerate transformers nvidia-ml-py torchvision huggingface_hub rich opencv-python
!pip install -q mediapy av imageio moviepy einops omegaconf timm safetensors sentencepiece rotary-embedding-torch tiktoken transformers_stream_generator lpips


In [ ]:
# [3] Tải mã nguồn SeedVR và vá lỗi tương thích môi trường Colab
%cd /content/upscale_video
!rm -rf seedvr_core
!git clone https://github.com/ByteDance-Seed/SeedVR.git seedvr_core

# Chạy script vá lỗi tự động (thay thế read_video bằng OpenCV + PyTorch & cài đặt Color Fix)
!python scripts/patch_seedvr_colab.py seedvr_core

# Áp dụng bản vá tối ưu bộ nhớ RoPE (giảm VRAM từ 4GB xuống 10KB chống OOM)
!python scripts/apply_rope_mem_patch.py

# Cài đặt FlashAttention (nếu hỗ trợ GPU compute capability >= 8.0, ngược lại Colab T4 sẽ tự dùng PyTorch SDPA)
!pip install -q flash_attn==2.5.9.post1 --no-build-isolation || echo "Colab Tesla T4 tự động dùng PyTorch SDPA mượt mà"


In [ ]:
# [4] Tải trọng số mô hình SeedVR2-3B từ Hugging Face vào ckpts/
import os, shutil
from huggingface_hub import snapshot_download

ckpt_dir = "/content/upscale_video/ckpts"
os.makedirs(ckpt_dir, exist_ok=True)
print("Đang tải trọng số SeedVR2-3B...")
snapshot_download(
    repo_id="ByteDance-Seed/SeedVR2-3B",
    local_dir=ckpt_dir,
    local_dir_use_symlinks=False,
    resume_download=True
)
os.makedirs("/content/upscale_video/seedvr_core", exist_ok=True)
seedvr_ckpts = "/content/upscale_video/seedvr_core/ckpts"
if not os.path.exists(seedvr_ckpts):
    try:
        os.symlink(ckpt_dir, seedvr_ckpts)
    except Exception:
        shutil.copytree(ckpt_dir, seedvr_ckpts, dirs_exist_ok=True)
print("Hoàn tất tải trọng số SeedVR2-3B!")


In [ ]:
# [5] Chạy thử nghiệm benchmark SeedVR2-3B trên 3 clip chuẩn (clip_A, clip_B, clip_C)
# Chạy pipeline tối ưu bộ nhớ Alternating Offload và Tiled VAE (Peak VRAM chỉ 7.47 GB trên Tesla T4)
!python scripts/run_eval_seedvr2.py


In [ ]:
# [6] Kết xuất video so sánh chi tiết mặt (Face Zoom) và toàn khung (Full)
!python scripts/eval_compare.py --models seedvr2 ccsr nomosuni

In [ ]:
# [7] Đóng gói kết quả và tải về máy cục bộ
!zip -r seedvr2_eval_results.zip eval/results/seedvr2/ eval/comparisons/
from google.colab import files
files.download('seedvr2_eval_results.zip')

In [ ]:
# [8] (Tùy chọn) Khởi chạy giao diện WebUI (Gradio) phục chế video trực quan
# Tạo đường link công khai https://xxx.gradio.live để truy cập từ trình duyệt
%cd /content/upscale_video
!python colab_app.py --share


In [ ]:
# [9] (Tùy chọn) Upscale video bất kỳ bằng SeedVR2-3B qua CLI
# Hỗ trợ tự động chia chunk và giữ nguyên âm thanh gốc
!python scripts/upscale_seedvr2.py -i /content/upscale_video/eval/clips/clip_A.mp4 -o /content/upscale_video/eval/results/seedvr2/output_custom.mp4
